In [1]:
import re
import os
import json
import calendar
from dateutil import parser
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor

import requests
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt
from datetime import datetime, timezone
from scipy.interpolate import RBFInterpolator
from scipy.interpolate import PchipInterpolator

from api_client import TradingDeskAPI
from options import OptionSurface, Deribit, OKX, Bybit
from scanner import MarketScanner

In [12]:
load_dotenv(r"D:/OneDrive/Trading/Prediction Markets/Moreton Capital/.env")
BASE_URL = os.getenv("BASE_URL")
USER_EMAIL = os.getenv("USER_EMAIL")
USER_PASSWORD = os.getenv("USER_PASSWORD")
print(BASE_URL)

target_expiry_str = "25DEC26"
target_expiry = datetime.strptime(target_expiry_str, "%d%b%y").strftime("%Y%m%d")

currencies = ["BTC", "ETH"]

s = OptionSurface()

deribit = Deribit(currencies=currencies, target_expiry=target_expiry)
okx = OKX(currencies=currencies, target_expiry=target_expiry)
bybit = Bybit(currencies=currencies, target_expiry=target_expiry)

s.initialize(currencies=currencies, exchanges=[deribit, okx, bybit])

https://alphasignal-dev.moretoncp.com
spot: 65151.0 volume24h: 0.4219
spot: 1923.7 volume24h: 3.7817
spot: 65169.9 volume24h: 976.71520009
spot: 1924.0 volume24h: 20654.694437
spot: 65174.5 volume24h: 2791.203945
spot: 1923.93 volume24h: 19162.50761


In [3]:
api = TradingDeskAPI(base_url=BASE_URL, email=USER_EMAIL, password=USER_PASSWORD)

markets = api.get_markets(limit=10000, liquidity_num_min=10000, volume_num_min=5000)

all_markets_df = pd.DataFrame(markets)

all_markets_df.head()

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,positionIds,negRiskMarketID,oneHourPriceChange,sportsMarketType,marketMetadata,clobRewards,gameId,eventStartTime,oneYearPriceChange,line
0,1343228,"Will Bitcoin dip to $5,000 by December 31, 2026?",0xe681a6326237f3b17ce8622728b6cd104281dfe4d2b3...,will-bitcoin-dip-to-5000-by-december-31-2026-j...,,2027-01-01T05:00:00Z,99872.66806,2026-02-05T22:18:33.034Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,3257358,"Will Bitcoin dip to $55,000 in August?",0x4d250b591dd7b7ffec3a518c1c83fc757f91d801762e...,will-bitcoin-dip-to-55k-in-august-2026,NaN,2026-09-01T04:00:00Z,99743.8069,2026-08-01T05:07:38Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1126504,Will the US acquire part of Greenland in 2026?,0x890fc3ba40458db0b67560691aa1597344c2a0560d18...,will-the-us-acquire-any-part-of-greenland-in-2026,,2026-12-31T00:00:00Z,99718.6542,2026-01-07T04:34:36.508662Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3150717,Will Kilmarnock FC win on 2026-08-09?,0x3c1aede701c4ecaf13ef91a06c0b252d31304244ba44...,scop-kil-cel-2026-08-09-kil,https://spfl.co.uk/,2026-08-09T12:30:00Z,99529.94474,2026-07-28T09:00:13Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,[104373446906527589101605569668810555903104862...,0xd5d65ce0fc84336820455e251969aff44ebb971f980d...,-0.0040,moneyline,"{'opticOddsFixtureId': '20260809D3BBB790', 'op...",NaN,NaN,NaN,NaN,NaN
4,1361085,Will Cincinnati Bengals win the 2027 NFL AFC C...,0x893eb655d409b75108367f2f8c6196a1d48f7e0fdd49...,will-cincinnati-bengals-win-the-2027-nfl-afc-c...,,2027-01-25T00:00:00Z,99098.10948,2026-02-09T22:33:51.244813Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,0x5337e0717e2659ac785da80b6a875d9839058c4c4e2a...,-0.0005,NaN,NaN,"[{'id': '852834', 'conditionId': '0x893eb655d4...",NaN,NaN,NaN,NaN


In [13]:
BTC_KEYWORDS = [
    "bitcoin",
    "btc",
    "xbt"
]

ETH_KEYWORDS = [
    "ethereum",
    " eth "
]

scanner = MarketScanner(api=api, BASE_URL=BASE_URL, USER_EMAIL=USER_EMAIL, USER_PASSWORD=USER_PASSWORD)
markets_df, opportunities_df = scanner.scan_market(markets_df=all_markets_df, s=s, KEYWORDS=BTC_KEYWORDS)

BTC
iv 0.4865314376901525
p_touch_above: 1.0 p_touch_below: 0.0
buy_yes_ev: -0.02030473
sell_yes_ev: 0.01676268
buy_no_ev: 0.01676268000000003
sell_no_ev: -0.020304729999999993
buy_yes_kelly: -0.010362778417823739
sell_yes_kelly: 0.5
buy_no_kelly: 0.5
sell_no_kelly: -0.010362778417823735
BTC
iv 0.43625017555198975
p_touch_above: 1.0 p_touch_below: 0.1091
buy_yes_ev: 0.023947999999999983
sell_yes_ev: -0.04365699999999999
buy_no_ev: -0.04365700000000008
sell_no_ev: 0.023947999999999983
buy_yes_kelly: 0.013088513064465344
sell_yes_kelly: -0.3335498067020154
buy_no_kelly: -0.3335498067020165
sell_no_kelly: 0.013088513064465344
BTC
iv 0.391210355635651
p_touch_above: 0.71573 p_touch_below: 1.0
buy_yes_ev: -0.028066999999999953
sell_yes_ev: -0.009842000000000017
buy_no_ev: -0.009842000000000017
sell_no_ev: -0.02806700000000001
buy_yes_kelly: -0.05477492457153106
sell_yes_kelly: -0.006971360895779513
buy_no_kelly: -0.006971360895779513
sell_no_kelly: -0.05477492457153118
BTC
iv 0.506113545847

In [14]:
opportunities_df

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,buy_yes_fee,sell_yes_fee,buy_no_fee,sell_no_fee,buy_yes_kelly,sell_yes_kelly,buy_no_kelly,sell_no_kelly,best_ev,best_action
19,2467210,"Will Bitcoin reach $70,000 by December 31, 2026?",0x7b9072e6a9cdcf022c4f098564e8ee612d544e179e4a...,will-bitcoin-reach-70000-by-december-31-2026-f...,,2027-01-01T05:00:00Z,96462.4803,2026-06-08T04:59:37.841638Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,-0.028067,-0.009842,-0.009842,-0.028067,-0.054775,-0.006971,-0.006971,-0.054775,0.014112,sell_yes_ev
38,701502,"Will Bitcoin dip to $45,000 by December 31, 2026?",0x024b68f77bfc019341ee3db8f57c103334e4b9430bba...,will-bitcoin-dip-to-45000-by-december-31-2026-...,,2027-01-01T05:00:00Z,94323.386,2025-11-24T19:07:15.386Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0.044898,-0.078523,-0.078523,0.044898,0.029231,-0.197904,-0.197904,0.029231,0.012012,buy_yes_ev
1,3257358,"Will Bitcoin dip to $55,000 in August?",0x4d250b591dd7b7ffec3a518c1c83fc757f91d801762e...,will-bitcoin-dip-to-55k-in-august-2026,NaN,2026-09-01T04:00:00Z,99743.8069,2026-08-01T05:07:38Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0.023948,-0.043657,-0.043657,0.023948,0.013089,-0.333550,-0.333550,0.013089,0.005152,buy_yes_ev
64,1343219,"Will Bitcoin dip to $20,000 by December 31, 2026?",0x23fb92bb72604ff35bfe591b5506de47f1bf6632ed55...,will-bitcoin-dip-to-20000-by-december-31-2026-...,,2027-01-01T05:00:00Z,91866.36926,2026-02-05T22:18:53.032Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0.005826,-0.018553,-0.018553,0.005826,0.003040,-0.321012,-0.321012,0.003040,0.002624,sell_no_ev
0,1343228,"Will Bitcoin dip to $5,000 by December 31, 2026?",0xe681a6326237f3b17ce8622728b6cd104281dfe4d2b3...,will-bitcoin-dip-to-5000-by-december-31-2026-j...,,2027-01-01T05:00:00Z,99872.66806,2026-02-05T22:18:33.034Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,-0.020305,0.016763,0.016763,-0.020305,-0.010363,0.500000,0.500000,-0.010363,0.001305,sell_no_ev
88,3257368,"Will Bitcoin dip to $42,500 in August?",0x98e5997750cfc27c47ad3b4f4278a01935bed1edfcec...,will-bitcoin-dip-to-42pt5k-in-august-2026,NaN,2026-09-01T04:00:00Z,90117.55339,2026-08-01T05:07:38Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,-0.003303,0.000986,0.000986,-0.003303,-0.001669,0.058841,0.058841,-0.001669,0.000693,sell_no_ev


In [ ]:
def build_positions(fills_df):

    if fills_df.empty:
        return pd.DataFrame()

    fills = fills_df.copy()

    fills["signed_shares"] = np.where(
        fills["side"].str.upper() == "BUY",
        fills["shares"],
        -fills["shares"]
    )

    positions = (
        fills
        .groupby(
            ["condition_id", "token_id", "outcome"],
            as_index=False
        )
        .agg(
            shares=("signed_shares", "sum")
        )
    )

    positions = positions[
        positions["shares"].abs() > 1e-9
    ]

    return positions

In [ ]:
# get all dfs
DATA_DIR = f"data"

orders_df = pd.read_parquet(f"{DATA_DIR}/orders.parquet")
fills_df = pd.read_parquet(f"{DATA_DIR}/fills.parquet")
positions_df = pd.read_parquet(f"{DATA_DIR}/positions.parquet")
realized_pnl_df = pd.read_parquet(f"{DATA_DIR}/realized_pnl.parquet")
equity_df = pd.read_parquet(f"{DATA_DIR}/equity.parquet")

In [ ]:
# Inventory Management Step

# buys = trades_df[trades_df.action == "BUY"]
# sells = trades_df[trades_df.action == "SELL"]

# position = (buys.shares.sum() - sells.shares.sum())

def format_signal(best_action, row, pos, current_bid, hold_ev, exit_ev, mtm_pnl, EXIT_THRESHOLD):

    print("============================================================")
    print(f'{"EXIT" if best_action == "exit" else "HOLD"} SIGNAL')
    print("============================================================")

    print(f"Market:\n{row.question}\n\n")
    print(f"Position:\n{pos.outcome}\n\n")
    print(f"Shares:\n{pos.shares}\n\n")
    print(f"Current Bid:\n{current_bid}\n\n")
    print(f"Exit EV:\n{exit_ev}\n\n")
    print(f"Hold EV:\n{hold_ev}\n\n")
    print(f"MTM Pnl:\n{mtm_pnl}\n\n")
    print(f"Exit_threshold:\n{EXIT_THRESHOLD}")


EXIT_THRESHOLD = -0.02
 
positions = api.get_positions()

for pos in positions:

    condition_id = pos["condition_id"]
    row = markets_df.loc[markets_df["condition_id"] == condition_id]

    if pos.outcome == "Yes": # no shorting in polymarket
        current_bid = row.yes_bid
        hold_ev = row.buy_yes_ev
        exit_ev = row.sell_yes_ev

    elif pos.outcome == "No":
        current_bid = row.no_bid
        hold_ev = row.buy_no_ev
        exit_ev = row.sell_no_ev

    if hold_ev < EXIT_THRESHOLD:
        best_action = "exit"

    else:
        best_action = "hold"

    size = pos.shares

    entry_price = pos.entry_price

    mtm_pnl = (current_bid - entry_price) * size

    format_signal(best_action, row, pos, current_bid, hold_ev, exit_ev, mtm_pnl, EXIT_THRESHOLD)

    if best_action == "exit":
        confirm_order = (input("Place this order? Type YES to confirm: ") == "YES")

        if confirm_order == True:

            price_tick = api.get_tick_size(row["token_id"])

            order = api.place_limit_order(
                token_id=row.token_id,
                side="SELL",
                price=api.round_to_tick(current_bid, price_tick),
                size=api.normalize_size(size),
                order_type="GTC",
            )

            print("\nORDER SUBMITTED\n\n")
            print(order)

            orders_df.loc[len(orders_df)] = {
                "order_id": order["clob_order_id"],
                "condition_id": condition_id,
                "token_id": row.token_id,
                "outcome": pos.outcome,
                "side": "SELL",
                "price": api.round_to_tick(current_bid, price_tick),
                "requested_size": api.normalize_size(size),
                "order_type": "GTC",
                "status": "OPEN",
                "created_at": datetime.now(timezone.utc),
                "cancelled_at": None,
            }

            print("\nORDER RECORDED\n\n")

        else:
            print("\nSKIPPING ORDER\n\n")

In [17]:
orders_df = pd.read_parquet("data/orders.parquet")

In [18]:
orders_df

,order_id,condition_id,token_id,outcome,side,price,requested_size,order_type,status,created_at,cancelled_at
0,YES,1,nan,YES,YES,YES,1.0,YES,YES,YES,YES


In [ ]:
def close_position(condition_id):

    positions = api.get_positions()

    pos = positions.find(condition_id)

    order = api.close_position(pos)

    # order_id
    # condition_id
    # token_id
    # outcome
    # side
    # price
    # requested_size
    # status
    # created_at
    # cancelled_at

    # order  = {"name": "Charlie", "age": 35}
    orders_df.loc[len(orders_df)] = order

    print("placed order to close position")

In [ ]:
# fill_id
# order_id
# condition_id
# token_id
# outcome
# side
# price
# shares
# fee
# timestamp
fill = {}

fills_df.loc[len(fills_df)] = fill

position = sum(
    fills_df.shares if fills_df.side == "BUY" else -fills_df.shares
    for fill in fills_df
)

In [15]:
ORDER_COLUMNS = [
    "order_id",
    "condition_id",
    "token_id",
    "outcome",
    "side",
    "price",
    "requested_size",
    "order_type",
    "status",
    "created_at",
    "cancelled_at",
]

df = []

df.append({
        "order_id": "YES",
        "condition_id": 1,
        "token_id": "nan",
        "outcome": "YES",
        "side": "YES",
        "price": "YES",
        "requested_size": 1.0,
        "order_type": "YES",
        "status": "YES",
        "created_at": "YES",
        "cancelled_at": "YES",
    })

df = pd.DataFrame(df)

DATA_DIR = f"data"
os.makedirs(DATA_DIR, exist_ok=True)
filename = f"{DATA_DIR}/orders.parquet"

df.to_parquet(filename, engine="fastparquet", index=False)

print('df saved at: ', filename)

df saved at:  data/orders.parquet


In [ ]:
# fill_id
# condition_id
# token_id
# outcome
# shares
# entry_price
# exit_price
# gross_pnl
# fees
# net_pnl
# entry_time
# exit_time
# holding_time

realized_pnl_df.loc[len(realized_pnl_df)] = row

In [ ]:
# timestamp
# cash
# market_value
# equity
# realized_pnl
# unrealized_pnl
# daily_return

# return_t = equity_t / equity_{t-1} - 1

equity_df.loc[len(equity_df)] = row

In [ ]:
# New Opportunities Step

def format_buy_signal(trade, outcome, best_action, normalized_size, current_ask,
                      unrealized_pnl, bankroll, MIN_EV, MAX_POSITION):

    print("============================================================")
    print(f"BUY SIGNAL")
    print("============================================================")

    print(f"Market:\n{trade.question}\n\n")
    print(f"Outcome:\n{outcome}\n\n")
    print(f"Best Action:\n{best_action}\n\n")
    print(f"Recommended Size:\n{normalized_size}\n\n")
    print(f"Current Ask:\n{current_ask}\n\n")
    print(f"Buy Yes EV:\n{trade.buy_yes_ev}\n\n")
    print(f"Sell No EV:\n{trade.sell_no_ev}\n\n")
    print(f"Buy No EV:\n{trade.buy_no_ev}\n\n")
    print(f"Sell Yes EV:\n{trade.sell_yes_ev}\n\n")
    print(f"Unrealized Pnl:\n{unrealized_pnl}\n\n")
    print(f"Bankroll:\n{bankroll}")
    print(f"MIN EV threshold:\n{MIN_EV}")
    print(f"MIN Position threshold:\n{MAX_POSITION}")

MIN_EV = 0.01   # require 1% edge, default = 0
bankroll = 100
MAX_POSITION = 0.05

positions = []

for _, trade in opportunities_df.iterrows():

    if trade.best_ev < MIN_EV:
        print(f"Current trade is less than required ev ({MIN_EV}), skipping")
        continue

    # usually cannot short, this only happens when im closing positions
    if trade.best_action == "buy_yes_ev" or trade.best_action == "sell_no_ev":

        ev = trade.buy_yes_ev
        token = trade.yes_token
        outcome = "YES"
        kelly = trade.buy_yes_kelly
        current_ask = trade.yes_ask

    elif trade.best_action == "buy_no_ev" or trade.best_action == "sell_yes_ev":

        ev = trade.buy_no_ev
        token = trade.no_token
        outcome = "NO"
        kelly = trade.buy_no_kelly
        current_ask = trade.no_ask

    dollars = bankroll * kelly

    dollars = min(dollars, bankroll * MAX_POSITION)

    position = api.get_conditional_balance(token)

    current_size = float(position["balance"])

    # inventory cap
    dollars = min(dollars, bankroll * MAX_POSITION - current_size * current_ask)

    if dollars <= 0:
        print("no more dollars to allocate for this trade position, skipping\n")
        continue

    size = dollars / current_ask

    normalized_size = api.normalize_size(size)

    unrealized_pnl = normalized_size * ev

    format_buy_signal(trade, outcome, best_action, normalized_size, current_ask,
                      unrealized_pnl, bankroll, MIN_EV, MAX_POSITION)

    confirm_order = (input("Place this order? Type YES to confirm: ") == "YES")
    
    if confirm_order == True:

        price_tick = api.get_tick_size(token)
        price = api.round_to_tick(current_bid, price_tick)

        order = api.place_limit_order(
            token_id=token,
            side="BUY",
            price=price,
            size=normalized_size,
            order_type="GTC",
        )

        print("\nORDER SUBMITTED\n\n")
        print(order)

        orders_df.loc[len(orders_df)] = {
            "order_id": order["clob_order_id"],
            "condition_id": condition_id,
            "token_id": token,
            "outcome": outcome,
            "side": "BUY",
            "price": price,
            "requested_size": normalized_size,
            "order_type": "GTC",
            "status": "OPEN",
            "created_at": datetime.now(timezone.utc),
            "cancelled_at": None,
        }

        print("\nORDER RECORDED\n\n")

    else:
        print("\nSKIPPING ORDER\n\n")

In [ ]:
def sync_orders(api, orders_df):

    api_orders = api.get_open_orders()

    for order in api_orders:

        order_id = order["id"]

        if order_id not in orders_df["order_id"].values:

            orders_df.loc[len(orders_df)] = {
                "order_id": order_id,
                "condition_id": order.get("market"),
                "token_id": order.get("asset_id"),
                "outcome": None,
                "side": order.get("side"),
                "price": order.get("price"),
                "requested_size": order.get("original_size"),
                "status": order.get("status"),
                "created_at": order.get("created_at"),
                "cancelled_at": None
            }

    return orders_df

In [ ]:
def sync_fills(api, fills_df):

    trades = api.get_trades()

    for trade in trades:

        fill_id = trade["id"]

        if fill_id not in fills_df["fill_id"].values:

            fills_df.loc[len(fills_df)] = {
                "fill_id": trade["id"],
                "order_id": trade.get("order_id"),
                "condition_id": trade["market"],
                "token_id": trade["asset_id"],
                "outcome": trade.get("outcome"),
                "side": trade["side"],
                "price": trade["price"],
                "shares": trade["size"],
                "fee": trade.get("fee", 0),
                "timestamp": trade["timestamp"]
            }

    return fills_df

def sync_positions():
    total_size = current_size + normalized_size
    
    new_entry_price = (current_size * entry_price + normalized_size * buy_price) / total_size

    fills_df.append({
        "condition_id": trade.conditionId,
        "token_id": token_id,
        "outcome": side,
        "size": shares,
        "entry_price": price,
        "entry_time": datetime.now()
    })

    fills.loc[
    fills.token_id == token,
    "shares"
] += new_shares

    

In [ ]:
def save(df):

    date = datetime.now().strftime("%Y%m%d")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    DATA_DIR = f"data/{date}"
    os.makedirs(DATA_DIR, exist_ok=True)
    filename = f"{DATA_DIR}/markets_{timestamp}.parquet"

    df.to_parquet(filename, engine="fastparquet", index=False)

    print('df saved at: ', filename)

save(btc_df)

In [ ]:
df = []

df.append({
        "condition_id": 1,
        "token_id": "nan",
        "outcome": "YES",
        "size": 1.0,
        "entry_price": 0.40,
        "entry_time": datetime.now(),
        "mtm_pnl": 0.0,
        "realized_pnl": 0.0
    })

df = pd.DataFrame(df)

DATA_DIR = f"data"
os.makedirs(DATA_DIR, exist_ok=True)
filename = f"{DATA_DIR}/positions.parquet"

df.to_parquet(filename, engine="fastparquet", index=False)

filename = f"{DATA_DIR}/trades.parquet"

df.to_parquet(filename, engine="fastparquet", index=False)

print('df saved at: ', filename)

In [ ]:
ENTRY_THRESHOLD = 0.03
EXIT_THRESHOLD = 0.01

edge = 0.031

if edge > ENTRY_THRESHOLD:
    pass
    #sell_yes()

if edge < EXIT_THRESHOLD:
    pass
    #close_position()

In [ ]:
#                  Polymarket    Model (exchanges: deribit, bybit, okx)    Edge

# 80k Dec-26        22%          20%       +2%
# 90k Dec-26        14%          12%       +2%
# 100k Dec-26        9.5%         5.9%     +3.6%
# 110k Dec-26        7%           4%       +3%
# 120k Dec-26        5%           2%       +3%

In [ ]:
while True:

    # 1. Manage inventory
    positions = api.get_positions()

    for pos in positions:

        market = api.get_market(pos.condition_id)

        if pos.outcome == "Yes":
            hold_ev = market.buy_yes_ev
            exit_ev = market.sell_yes_ev

        else:
            hold_ev = market.buy_no_ev
            exit_ev = market.sell_no_ev

        if hold_ev < EXIT_THRESHOLD:
            api.close_position(pos)


    # 2. Find new opportunities
    opportunities = scanner.scan_market()

    for trade in opportunities:

        if trade.best_ev > MIN_EV:
            api.place_limit_order(...)


In [ ]:
# Keep trading journal

# Your thesis.
# Why you think the market is mispriced.
# Position size.
# Exit criteria.
# What actually happened.

In [ ]:
# Stage 2 — Semi-automated execution

# The bot does:

# find opportunities
# calculate size
# prepare orders

# You approve:

# BUY YES
# Market: BTC above $150k
# Price: 0.43
# Size: $500
# Expected edge: +8%

# Click "confirm".

# This is useful because prediction markets can have:

# ambiguous wording
# resolution risks
# sudden news events

In [ ]:
# Day 1 — API connection + market ingestion

# Goal:

# Can I pull markets automatically?

# Build:

# get_markets()

# Output:

# {
#  "condition_id": "...",
#  "question": "Will X happen?",
#  "tokens": [
#     {
#       "token_id": "YES",
#       "price": 0.42
#     },
#     {
#       "token_id": "NO",
#       "price": 0.58
#     }
#  ]
# }

# Store:

# markets

# condition_id
# question
# yes_token
# no_token
# created_time
# Day 2 — Historical snapshots

# Goal:

# Can I reconstruct what the market looked like yesterday?

# Every 5 minutes:

# while True:

#     markets = api.get_markets()

#     for m in markets:
#         database.save_snapshot(m)

#     sleep(300)

# Database:

# market_snapshots

# timestamp
# condition_id
# yes_price
# no_price
# volume
# Day 3 — Order book collection

# Now collect microstructure data.

# For selected markets:

# get_orderbook(token_id)

# Store:

# orderbook_snapshots

# timestamp

# token_id

# best_bid
# best_ask

# bid_depth
# ask_depth

# Calculate:

# Spread
# spread=ask−bid

# Example:

# Bid:
# 0.42

# Ask:
# 0.46

# Spread:
# 4 cents
# Day 4 — Build your scanner

# Your first scanner should be dumb but useful.

# Signal 1: Large moves
# if abs(price_change_24h) > 0.10:
#     flag()

# Example:

# AI model release

# Yesterday:
# 35%

# Today:
# 52%

# Move:
# +17%
# Signal 2: Liquidity opportunities
# if spread > 0.08:
#     flag()
# Signal 3: Volume spikes
# if volume_today > 5 * average_volume:
#     flag()

# Your output:

# TOP MARKETS TO REVIEW

# 1.
# Question:
# Will Fed cut rates?

# Price:
# 42%

# 24h move:
# +12%

# Reason:
# Large movement


# 2.
# Question:
# Will Company X acquire Y?

# Price:
# 33%

# Spread:
# 11 cents

# Reason:
# Wide market
# Day 5 — Add your probability workflow

# Do NOT automate this yet.

# Create a manual table:

# trade_journal.csv

# market,current_price,my_probability,edge,reason
# Fed cut,0.42,0.55,0.13,"Inflation falling"
# AI launch,0.35,0.45,0.10,"Company comments"

# The key question:

# Market probability:
# 42%

# My probability:
# 55%

# Difference:
# +13%
# Day 6 — Paper trading

# Before your C++ engine touches anything:

# Create:

# paper_buy(
#     market,
#     price,
#     size
# )

# Track:

# Position:
# YES Fed cut

# Entry:
# 42c

# Size:
# $100

# Current:
# 48c

# P&L:
# +$14
# Day 7 — Analytics

# Calculate:

# Return
# profit / capital
# Drawdown

# Largest loss from peak.

# Sortino

# Track:

# returns
# negative returns only

# Your output:

# Paper Portfolio

# Trades:
# 18

# Win rate:
# 61%

# Return:
# +8.4%

# Max drawdown:
# -2.1%

# Sortino:
# 2.4

In [ ]:
{'id': '701496', 'question': 'Will Bitcoin reach $100,000 by December 31, 2026?', 'conditionId': '0xdaa4866bae18be58c5a79d2aeeffd035ec78f1bb49dbd88f72993997778a990f', 'slug': 'will-bitcoin-reach-100000-by-december-31-2026-571-361-361', 
 'resolutionSource': '', 'endDate': '2027-01-01T05:00:00Z', 'liquidity': '94083.1351', 'startDate': '2025-11-24T19:07:17.691Z', 'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 
 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 
'description': 'This market will immediately resolve to "Yes" if any Binance 1 minute candle for Bitcoin (BTC/USDT) between November 24, 2025, 14:00 and December 31, 2026, 23:59 in the ET timezone has a final "High" price equal to or greater than the price specified in the title. Otherwise, this market will resolve to "No."\n\nThe resolution source for this market is Binance, specifically the BTC/USDT "High" prices available at https://www.binance.com/en/trade/BTC_USDT, with the chart settings on "1m" for one-minute candles selected on the top bar.\n\nPlease note that the outcome of this market depends solely on the price data from the Binance BTC/USDT trading pair. Prices from other exchanges, different trading pairs, or spot markets will not be considered for the resolution of this market.', 
'outcomes': '["Yes", "No"]', 'outcomePrices': '["0.095", "0.905"]', 'volume': '2370982.5583320004', 'active': True, 'closed': False, 'marketMakerAddress': '', 'createdAt': '2025-11-24T18:55:12.725029Z', 'updatedAt': '2026-08-02T10:54:52.526927Z', 
'new': False, 'featured': False, 'submitted_by': '0x91430CaD2d3975766499717fA0D66A78D814E5c5', 'archived': False, 'resolvedBy': '0x65070BE91477460D8A7AeEb94ef92fe056C2f2A7', 'restricted': True, 'groupItemTitle': '↑ 100,000', 'groupItemThreshold': '13', 
'questionID': '0x3c9be67d4b90291760ac3bffc1f9470a1966e5c1f3e99131333170e3469bd023', 'enableOrderBook': True, 'orderPriceMinTickSize': 0.01, 'orderMinSize': 5, 'volumeNum': 2370982.5583320004, 'liquidityNum': 94083.1351, 'endDateIso': '2027-01-01', 
'startDateIso': '2025-11-24', 'hasReviewedDates': True, 'volume24hr': 1365.610655, 'volume1wk': 66640.791618, 'volume1mo': 228733.97684700004, 'volume1yr': 2370982.5583320004, 
'clobTokenIds': '["56078938060096976448086754249497300447360333783952000147427828224794011030104", "11291662904897713174667903388388696640643610556195928998276904135282270136756"]', 
'comboStatus': 'disabled', 'umaBond': '500', 'umaReward': '5', 'volume24hrClob': 1365.610655, 'volume1wkClob': 66640.791618, 'volume1moClob': 228733.97684700004, 'volume1yrClob': 2370982.5583320004, 'volumeClob': 2370982.5583320004, 
'liquidityClob': 94083.1351, 'makerBaseFee': 1000, 'takerBaseFee': 1000, 'customLiveness': 0, 'acceptingOrders': True, 'negRisk': False, 'negRiskRequestID': '', 
'events': [{'id': '89502', 'ticker': 'what-price-will-bitcoin-hit-before-2027', 'slug': 'what-price-will-bitcoin-hit-before-2027', 
            'title': 'What price will Bitcoin hit in 2026?', 'description': 'What price will Bitcoin hit before 2027?  ', 
            'resolutionSource': '', 'startDate': '2025-11-24T19:07:12.848Z', 'creationDate': '2025-11-24T19:13:13.705687Z', 'endDate': '2027-01-01T05:00:00Z', 
            'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png', 
            'active': True, 'closed': False, 'archived': False, 'new': False, 'featured': False, 'restricted': True, 'liquidity': 2695251.24076, 'volume': 50935482.262915, 
            'openInterest': 9503925.568983998, 'createdAt': '2025-11-24T18:55:05.597959Z', 'updatedAt': '2026-08-02T10:55:09.654318Z', 'competitive': 0.9999750006249843, 
            'volume24hr': 130499.32620200001, 'volume1wk': 2062901.9714630004, 'volume1mo': 7099616.726362999, 'volume1yr': 49102712.92022599, 'enableOrderBook': True, 
            'liquidityClob': 2695251.24076, 'negRisk': False, 'commentCount': 0, 'series': [{'id': '10016', 'ticker': 'bitcoin-hit-price-monthly', 'slug': 'bitcoin-hit-price-monthly', 
                                                                                        'title': 'Bitcoin Hit Price Monthly', 'seriesType': 'single', 'recurrence': 'monthly', 
                                                                                        'image': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/bitcoin+colors.jpeg', 
                                                                                        'icon': 'https://polymarket-upload.s3.us-east-2.amazonaws.com/bitcoin+colors.jpeg', 
                                                                                        'active': True, 'closed': False, 'archived': False, 'featured': False, 'restricted': True, 
                                                                                        'createdAt': '2025-01-31T22:03:50.00441Z', 'updatedAt': '2026-08-02T10:55:28.185077Z', 
                                                                                        'volume24hr': 601000.637578, 'volume': 51492794.133888, 'liquidity': 3438609.2653, 'commentCount': 6318, 
                                                                                        'requiresTranslation': False}], 
            'cyom': False, 'showAllOutcomes': True, 'showMarketImages': False, 'enableNegRisk': False, 'automaticallyActive': True, 'seriesSlug': 'bitcoin-hit-price-monthly', 
            'gmpChartMode': 'default', 'negRiskAugmented': False, 'estimateValue': True, 'cantEstimate': True, 'cumulativeMarkets': False, 'pendingDeployment': False, 'deploying': False, 
            'requiresTranslation': False, 'eventMetadata': {'context_requires_regen': True}, 'version': 'v1'}], 

'ready': False, 'funded': False, 'acceptingOrdersTimestamp': '2025-11-24T19:06:55Z', 
'cyom': False, 'competitive': 0.8590880780051975, 'pagerDutyNotificationEnabled': False, 'approved': True, 'clobRewards': [{'id': '418394', 'conditionId': '0xdaa4866bae18be58c5a79d2aeeffd035ec78f1bb49dbd88f72993997778a990f', 
                                                                                                                        'assetAddress': '0xc011a7e12a19f7b1f670d46f03b03f3342e82dfb', 'rewardsAmount': 0, 'rewardsDailyRate': 0.001, 
                                                                                                                        'startDate': '2026-06-03', 'endDate': '2500-12-31'}], 
'rewardsMinSize': 0, 'rewardsMaxSpread': 0, 'spread': 0.01, 'oneMonthPriceChange': -0.01, 'lastTradePrice': 0.09, 'bestBid': 0.09, 'bestAsk': 0.1, 'automaticallyActive': True, 
'clearBookOnStart': True, 'seriesColor': '', 'showGmpSeries': False, 'showGmpOutcome': False, 'manualActivation': False, 'negRiskOther': False, 'umaResolutionStatuses': '[]', 
'pendingDeployment': False, 'deploying': False, 'deployingTimestamp': '2025-11-24T19:06:23.727362Z', 'rfqEnabled': False, 'holdingRewardsEnabled': True, 'feesEnabled': True, 
'requiresTranslation': False, 'feeType': 'crypto_fees_v2', 'feeSchedule': {'exponent': 1, 'rate': 0.07, 'takerOnly': True, 'rebateRate': 0.2}, 'version': 'v1'}
["0.095", "0.905"]